# Documentation
**Author:** Spencer Ressel

**Created:** January 17th, 2023

***

This script is a compilation of functions that serve to recreate the figures of 
two papers, Wheeler and Hendon (2004) and Jiang et al. (2020).
The Wheeler and Hendon (2004) paper describes the creation of a real-time multivariate
MJO index (RMM), and splits the MJO into 8 phases based on the RMM index. 
The Jiang et al. (2020) paper is a broad overview of the MJO,
where the specific figure (Fig. 1) recreated is a composite of MJO-associated rainfall
anomalies during boreal winter for each MJO phase.

***

**Inputs:**     
* Global 2.5° x 2.5° resolution, daily timeseries in netCDF format:
    - precipitation from TRMM
    - outgoing longwave radiation (OLR) data from *Liebmann and Smith (1996)*
    - 200 hPa zonal and meridional wind from ERA5 reanalysis
    - 850 hPa zonal and meridional wind from ERA5 reanalysis
                
**Dependencies:**
* mjo_mean_state_diagnostics.py

# Imports

In [ ]:
import os
os.chdir('/home/disk/eos7/sressel/research/thesis-work/python/mjo_data_analysis/')

import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
)
logger = logging.getLogger(__name__)

# Data processing tools
import numpy as np
import scipy
import scipy.signal as signal
from scipy.optimize import curve_fit
import xarray as xr
import xeofs

import sys
sys.path.insert(0, '/home/disk/eos7/sressel/research/thesis-work/python/auxiliary_functions/')
# import ipynb.fs.full.mjo_mean_state_diagnostics as mjo
from auxiliary_functions.plotting_utils import modified_colormap
from auxiliary_functions import xarray_utils
from auxiliary_functions.mjo_mean_state_diagnostics import remove_annual_cycle, lanczos_bandpass_filter

# Plotting
from tqdm import tqdm
from matplotlib import pyplot as plt
from matplotlib import ticker as mticker
from matplotlib import colors as mcolors
from matplotlib.gridspec import GridSpec
from matplotlib.animation import FuncAnimation

# Cartopy
from cartopy import crs as ccrs
from cartopy import feature as cf
from cartopy import util as cutil
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter, LongitudeLocator, LatitudeLocator

# Seaborn
import seaborn as sns

# Set Physical Constants and Analysis Parameters

In [ ]:
# Set time bounds
TIME_MIN = '1974-06-01T00:00:00.000000000'
TIME_MAX = '2005-12-31T00:00:00.000000000'
# TIME_MAX = '2001-12-31T00:00:00.000000000'
# TIME_MIN = '1999-01-01T00:00:00.000000000'
# TIME_MIN = '1990-01-01T00:00:00.000000000'
# TIME_MAX = '1995-12-31T00:00:00.000000000'

missing_days = np.arange(np.datetime64("1978-03-17"), np.datetime64("1978-12-31"))

SAMPLING_FREQUENCY = 1

# Set latitude bounds
LATITUDE_SOUTH = -25
LATITUDE_NORTH = 25

# Set central longitude
CENTRAL_LONGITUDE = 160

# Set longitude bounds
LONGITUDE_MIN = 0
LONGITUDE_MAX = 360

# Cut-off periods for intraseasonal filtering
INTRASEASONAL_LOWCUT = 200
INTRASEASONAL_HIGHCUT = 20

# Seconds per day
SECONDS_PER_DAY = 24 * 3600

# Load Data

In [ ]:
variables_to_load = [
    # 'Precipitation',
    'Outgoing Longwave Radiation',
    # 'Zonal Wind',
    # 'Meridional Wind'
]

In [ ]:
logger.info("Load Variables")
# TRMM Precipitation
if 'Precipitation' in variables_to_load:
    logger.info("    Precipitation...")
    data_directory_precip = r"/home/disk/eos7/sressel/research/data/NASA/TRMM/"
    file_name_precip = "trmm_precipitation_daily_1998_2018.nc"
    data_precipitation = xr.open_dataset(
        data_directory_precip + file_name_precip, engine="netcdf4"
    )
    precipitation = data_precipitation['precipitation'].sortby('lat')

# NASA OLR (Liebmann and Smith 1996)
if 'Outgoing Longwave Radiation' in variables_to_load:
    logger.info("    Outgoing Longwave Radiation...")
    data_directory_olr = r"/home/disk/eos7/sressel/research/data/NOAA/"
    file_name_olr = "olr.day.mean.nc"
    data_olr = xr.open_dataset(data_directory_olr + file_name_olr, engine="netcdf4")
    outgoing_longwave_radiation = data_olr['olr'].sortby('lat')

# ERA5 Zonal Wind
if 'Zonal Wind' in variables_to_load:
    logger.info("    Zonal Wind...")
    data_directory_zonal_wind = r"/home/disk/p/sressel/eos7-link/research/data/ECMWF/ERA5/daily_data/25_degree_data"
    file_name_zonal_wind = "daily_25_degree_zonal_wind_1980_2018.nc"
    data_zonal_wind = xr.open_dataset(f"{data_directory_zonal_wind}/{file_name_zonal_wind}", engine="netcdf4"
    )

    zonal_wind = data_zonal_wind["u"].sortby('lat')
    zonal_wind_longitudes = zonal_wind["lon"].values
    zonal_wind_longitudes[zonal_wind_longitudes < 0] += 360
    zonal_wind["lon"] = zonal_wind_longitudes
    zonal_wind = zonal_wind.sortby(zonal_wind.lon)

    upper_level_zonal_wind = zonal_wind.sel(plev=200)
    lower_level_zonal_wind = zonal_wind.sel(plev=850)

# ERA5 Meridional Wind
if 'Meridional Wind' in variables_to_load:
    logger.info("    Meridional Wind...")
    data_directory_meridional_wind = r"/home/disk/eos7/sressel/research/data/ECMWF/ERA5/daily_data/25_degree_data"
    file_name_meridional_wind = (
        "daily_25_degree_meridional_wind_1980_2018.nc"
    )
    data_meridional_wind = xr.open_dataset(f"{data_directory_meridional_wind}/{file_name_meridional_wind}", engine="netcdf4"
    )

    meridional_wind = data_meridional_wind["v"].sortby('lat')
    meridional_wind_longitudes = meridional_wind["lon"].values
    meridional_wind_longitudes[meridional_wind_longitudes < 0] += 360
    meridional_wind["lon"] = meridional_wind_longitudes
    meridional_wind = meridional_wind.sortby(meridional_wind.lon)

    upper_level_meridional_wind = meridional_wind.sel(plev=200)
    lower_level_meridional_wind = meridional_wind.sel(plev=850)

time = outgoing_longwave_radiation.time
latitude = outgoing_longwave_radiation.lat
longitude = outgoing_longwave_radiation.lon


variables_dict = {
    # 'Precipitation' : precipitation,
    'Outgoing Longwave Radiation' : outgoing_longwave_radiation,
    # 'Upper Level Zonal Wind' : upper_level_zonal_wind,
    # 'Lower Level Zonal Wind' : lower_level_zonal_wind,
    # 'Upper Level Meridional Wind' : upper_level_meridional_wind,
    # 'Lower Level Meridional Wind' : lower_level_meridional_wind,
}
logger.info("Finished")

# Process Data

Detrend the data, remove the annual cycle and the first three harmonics (seasonal cycle), and filter the data on intraseasonal timescales

## Subset Data
Specifically select the data from times of interest and from tropical latitudes

### Perform subsetting

In [ ]:
logger.info("Subset variables")
variables_subset = {}
for variable in variables_dict:
    logger.info(f"    {variable}...")
    if variable != 'Precipitation':
        variables_subset[variable] = variables_dict[variable].copy(deep=True)
        variables_subset[variable] = variables_dict[variable].sel(
            time=slice(TIME_MIN, TIME_MAX),
            lat=slice(LATITUDE_SOUTH, LATITUDE_NORTH)
        )

if 'Precipitation' in variables_dict:
    variables_subset['Precipitation'] = variables_dict['Precipitation'].copy(deep=True).sel(
        time=slice('1999-01-01T00:00:00.000000000', '2018-12-31T00:00:00.000000000'),
        lat=slice(LATITUDE_SOUTH, LATITUDE_NORTH)
    )

latitude = latitude.sel(lat=slice(LATITUDE_SOUTH, LATITUDE_NORTH))
time = time.sel(time=slice(TIME_MIN, TIME_MAX))

logger.info("Finished")

### Plot subset data

In [ ]:
# Set plotting parameters
output_directory = "output/mjo-compositing/"
plt.style.use('default')
plt.rcParams.update({'font.size':24})
cmap_modified = modified_colormap('coolwarm', 'white', 0.1, 0.1)
# cmap_modified = 'BrBG'
coastline_width = 1

outgoing_longwave_radiation_anomalies = variables_subset['Outgoing Longwave Radiation'].stats.standardize(dim=['lat', 'lon'])
lower_level_zonal_wind_anomalies = variables_subset['Lower Level Zonal Wind'].stats.standardize(dim=['lat', 'lon'])
lower_level_meridional_wind_anomalies = variables_subset['Lower Level Meridional Wind'].stats.standardize(dim=['lat', 'lon'])

fig = plt.figure(figsize=(16,4))
gs = GridSpec(1, 2, width_ratios=[1, 0.02], figure=fig)
gs.update(top=.95, bottom=0.05, left=0.05, right=.95, hspace=0, wspace=0.05)

proj = ccrs.PlateCarree(central_longitude=-205)
data_crs = ccrs.PlateCarree()

ax = fig.add_subplot(gs[0], projection=proj)
cb_ax = fig.add_subplot(gs[1])


# Moisture Tendency
ax.set_title(f"Time-Mean Outgoing Longwave Radiation Anomalies and 850-hPa Zonal Winds", fontsize=20)

# Add cyclic point
cdata = xarray_utils.add_cyclic_point(
    outgoing_longwave_radiation_anomalies,
    dim='lon'
)

# Plot data
im = ax.contourf(
    cdata.lon, 
    cdata.lat, 
    cdata.mean(dim='time'), 
    transform=data_crs,
    cmap=cmap_modified,
    norm=mcolors.CenteredNorm(),
    levels=21
)

# Add colorbar
cbar = fig.colorbar(im, cax=cb_ax)
cbar.ax.tick_params(labelsize=20)
cbar.set_label(r'W m$^{-2}$')

arrow_spacing = 1
ax.quiver(
    lower_level_zonal_wind_anomalies.lon[::2*arrow_spacing],
    lower_level_zonal_wind_anomalies.lat[::arrow_spacing],
    lower_level_zonal_wind_anomalies.mean(dim='time').values[::arrow_spacing, ::2*arrow_spacing],
    lower_level_meridional_wind_anomalies.mean(dim='time').values[::arrow_spacing, ::2*arrow_spacing],
    width=0.001,
    scale=500,
    transform=data_crs
)

ax.set_aspect('auto')
ax.set_xlabel('')
# ax.set_global()
ax.add_feature(cf.COASTLINE, lw=coastline_width)

gl = ax.gridlines(
    crs=proj,
    draw_labels=True,
    linewidth=1,
    color="gray",
    alpha=0.75,
    linestyle="-",
    zorder=15

)
gl.right_labels = False
gl.top_labels = False
gl.xlocator = mticker.FixedLocator(np.arange(-180,180,30))
# gl.xlocator = LongitudeLocator(30)
gl.xformatter = LongitudeFormatter()
gl.xlabel_style = {'fontsize':20}
gl.ylocator = mticker.FixedLocator(np.arange(-40,40,10))
gl.yformatter = LatitudeFormatter()
gl.ylabel_style = {'fontsize':20}

plt.show()
# plt.savefig(f"{output_directory}/time_mean_OLR_zonal_wind_anomalies.png", dpi=300, bbox_inches='tight')

## Remove mean & Detrend the data

In [ ]:
variables_detrended = {}

logger.info("Detrend data")
for variable_name, variable_data in variables_subset.items():
    logger.info(f"    {variable_name}...")
    variables_detrended[variable_name] = xr.zeros_like(variable_data)
    variables_detrended[variable_name][:] = signal.detrend(
        variable_data.stats.standardize(dim='time'), 
        axis=variable_data.get_axis_num('time'), 
        type='linear'
    )
logger.info("Finished")

## Remove the Annual Cycle

In [14]:
variables_deannualized = {}
variables_annual_cycle = {}
logger.info("Remove the annual cycle")

for variable_name, variable_data in variables_subset.items():
    logger.info(f"→ {variable_name}...")
    [           
        variables_deannualized[variable_name],
        variables_annual_cycle[variable_name],
    ] = remove_annual_cycle(variable_data)    

logger.info("Finished")

2026-05-26 09:43:55,455 [INFO] Remove the annual cycle
2026-05-26 09:43:55,457 [INFO] → Outgoing Longwave Radiation...
2026-05-26 09:45:44,861 [INFO] Finished


# Identify the MJO's spectral peak

There should be a coherency between zonal wind and outgoing longwave radiation, with a phase relationship of 90°. Minima in OLR correspond to maxima in convection and thus near-zero zonal winds

## Choose a single location

In [ ]:
# Specify location to find MJO signal
single_location_latitude = 0
single_location_longitude = 120

variables_single_location = {}
for variable in variables_dict:
    variables_single_location[variable] = variables_deannualized[variable].copy(deep=True)
    variables_single_location[variable] = variables_deannualized[variable].sel(
        lat=single_location_latitude,
        lon=single_location_longitude
    )

## Standardized time series

In [ ]:
def standardize_time_series(time_series):
    return (time_series - np.mean(time_series))/np.std(time_series)

In [ ]:
# Remove the mean and divide by the standard deviation
outgoing_longwave_radiation_standardized = standardize_time_series(
    variables_single_location['outgoing longwave radiation']
)
lower_level_zonal_wind_standardized = standardize_time_series(
    variables_single_location['lower level zonal wind']
)
upper_level_zonal_wind_standardized = standardize_time_series(
    variables_single_location['upper level zonal wind']
)

print('Check standardization')
print(f"Time Series Means:\nOLR: {outgoing_longwave_radiation_standardized.mean(dim='time').values:>10.2f} \n"
      + f"U850:  {lower_level_zonal_wind_standardized.mean(dim='time').values:>8.2f}\n"
      + f"U200:  {upper_level_zonal_wind_standardized.mean(dim='time').values:>8.2f}")
print(f"Time Series Standard Deviations:\nOLR: {outgoing_longwave_radiation_standardized.std(dim='time').values:>10.2f} \n"
      + f"U850:  {lower_level_zonal_wind_standardized.std(dim='time').values:>8.2f}\n"
      + f"U200:  {upper_level_zonal_wind_standardized.std(dim='time').values:>8.2f}")

## Calculate autocorrelation

In [ ]:
# Calculate the autocorrelation for a given lag
def calculate_autocorrelation(time_series, lag):
    standard_deviation = np.std(time_series)
    mean = np.mean(time_series)
    time_series = (time_series - mean)/standard_deviation
    
    autocorrelation = np.sum(
        (time_series[lag:]) 
      * (time_series[:-lag]),
        axis=0
    ) / ((len(time_series) - lag))
    
    return autocorrelation

In [ ]:
outgoing_longwave_radiation_autocorrelation = calculate_autocorrelation(
    outgoing_longwave_radiation_standardized, 
    1
)
lower_level_zonal_wind_autocorrelation = calculate_autocorrelation(
    lower_level_zonal_wind_standardized,
    1
)

upper_level_zonal_wind_autocorrelation = calculate_autocorrelation(
    upper_level_zonal_wind_standardized,
    1
)

print("Autocorrelations")
print(f"OLR:  {outgoing_longwave_radiation_autocorrelation.values:0.2f}")
print(f"U850: {lower_level_zonal_wind_autocorrelation.values:0.2f}")
print(f"U200: {upper_level_zonal_wind_autocorrelation.values:0.2f}")

## Compute power spectra

In [ ]:
# Specify window size
window_size = 256

# Calculate raw spectra

frequency_olr, outgoing_longwave_radiation_power_spectra = signal.welch(
    outgoing_longwave_radiation_standardized,
    nperseg=window_size,
    noverlap=window_size//2, 
    detrend='linear',
    fs=1
)

frequency_lower, lower_level_zonal_wind_power_spectra = signal.welch(
    lower_level_zonal_wind_standardized,
    nperseg=window_size,
    noverlap=window_size//2, 
    detrend='linear',
    fs=1
)

frequency_upper, upper_level_zonal_wind_power_spectra = signal.welch(
    upper_level_zonal_wind_standardized,
    nperseg=window_size,
    noverlap=window_size//2, 
    detrend='linear',
    fs=1
)

outgoing_longwave_radiation_power_normalized = outgoing_longwave_radiation_power_spectra/np.mean(outgoing_longwave_radiation_power_spectra)
lower_level_zonal_wind_power_normalized = lower_level_zonal_wind_power_spectra/np.mean(lower_level_zonal_wind_power_spectra)
upper_level_zonal_wind_power_normalized = upper_level_zonal_wind_power_spectra/np.mean(upper_level_zonal_wind_power_spectra)

## Fit red spectrum

In [ ]:
def calculate_red_spectrum(frequency, autocorrelation):
    red_spectrum = (1-autocorrelation**2)/(1-(2*autocorrelation*np.cos(frequency*2*np.pi))+autocorrelation**2)
    return red_spectrum

In [ ]:
p_critical = 0.95

# Fit red noise spectrum to data
[olr_parameters, olr_covariance] = curve_fit(
    calculate_red_spectrum, 
    frequency_olr, 
    outgoing_longwave_radiation_power_normalized,
    p0=(0.5)
)

outgoing_longwave_radiation_red_spectrum = calculate_red_spectrum(frequency_olr, olr_parameters[0])

[lower_parameters, lower_covariance] = curve_fit(
    calculate_red_spectrum, 
    frequency_lower, 
    lower_level_zonal_wind_power_normalized,
    p0=(0.5)
)
lower_level_zonal_wind_red_spectrum = calculate_red_spectrum(frequency_lower, lower_parameters[0])

[upper_parameters, upper_covariance] = curve_fit(
    calculate_red_spectrum, 
    frequency_lower, 
    upper_level_zonal_wind_power_normalized,
    p0=(0.5)
)
upper_level_zonal_wind_red_spectrum = calculate_red_spectrum(frequency_upper, upper_parameters[0])

# Estimate degrees of freedom
degrees_of_freedom_numerator = 2*len(outgoing_longwave_radiation_standardized)/window_size
degrees_of_freedom_denominator = len(outgoing_longwave_radiation_standardized)//2
f_critical = scipy.stats.f.ppf(p_critical, degrees_of_freedom_numerator, degrees_of_freedom_denominator)

## Plot Power Spectra

In [ ]:
# Specify periods to plot as xticks
periods = np.array([100, 32, 20,  15, 11.25, 9, 7.5, 6.4, 5.6, 5])
frequency_from_period = 1/periods

plt.style.use('bmh')
plt.rcParams.update({'font.size':18})
title_size = 24

gs = GridSpec(3, 1, height_ratios=(1, 1, 1))
gs.update(top=.95, bottom=0.05, left=0.05, right=.95, hspace=0.3, wspace=0.05)
f = plt.figure(figsize=(16,16))

ax = []
ax.append(f.add_subplot(gs[0]))
ax.append(f.add_subplot(gs[1]))
ax.append(f.add_subplot(gs[2]))

ax[0].set_title(
    f'Power spectra of Outgoing Longwave Radiation' +
    f' at the point {single_location_longitude:0.0f}°E, {single_location_latitude:0.0f}', 
    pad=15,    
    fontsize=title_size
)

# Plot OLR Power Spectrum
ax[0].plot(
    frequency_olr, 
    outgoing_longwave_radiation_power_normalized, 
    lw=4, 
    color='#348ABD', 
    label='OLR'
)

# OLR Red Spectrum Fit
ax[0].plot(
    frequency_olr, 
    outgoing_longwave_radiation_red_spectrum, 
    lw=4, 
    color='#A60628', 
    ls='-',
    label='OLR Red'
)

# OLR 95% confidence level
ax[0].plot(
    frequency_olr, 
    f_critical*outgoing_longwave_radiation_red_spectrum, 
    lw=4, 
    color='#A60628', 
    ls='--', 
    label='95%'
)
ax[0].set_ylim(0,7)


# Plot power spectrum of lower level zonal winds
ax[1].set_title(
    f'Power spectra of U850' +
    f' at the point {single_location_longitude:0.0f}°E, {single_location_latitude:0.0f}', 
    pad=15,
    fontsize=title_size
)

# Power spectrum
ax[1].plot(
    frequency_lower, 
    lower_level_zonal_wind_power_normalized, 
    lw=4, 
    color='#7A68A6', 
    label='U850'
)

# Red Spectrum Fit
ax[1].plot(
    frequency_lower, 
    lower_level_zonal_wind_red_spectrum, 
    lw=4, 
    color='#467821', 
    ls='-', 
    label='U850 Red'
)

# 95% confidence level
ax[1].plot(
    frequency_lower, 
    f_critical*lower_level_zonal_wind_red_spectrum, 
    lw=4, 
    color='#467821', 
    ls='--', 
    label='95%'
)
ax[1].set_ylim(0,9)


# Plot power spectrum of upper level zonal winds
ax[2].set_title(
    f'Power spectra of U200' +
    f' at the point {single_location_longitude:0.0f}°E, {single_location_latitude:0.0f}',         
    pad=15,        
    fontsize=title_size
)

# Power spectrum
ax[2].plot(
    frequency_upper, 
    upper_level_zonal_wind_power_normalized, 
    lw=4, 
    color='#7A68A6', 
    label='U200'
)

# Red Spectrum Fit
ax[2].plot(
    frequency_upper, 
    upper_level_zonal_wind_red_spectrum, 
    lw=4, 
    color='#467821', 
    ls='-', 
    label='U200 Red'
)

# 95% confidence level
ax[2].plot(
    frequency_upper, 
    f_critical*upper_level_zonal_wind_red_spectrum, 
    lw=4, 
    color='#467821', 
    ls='--', 
    label='95%'
)
ax[2].set_ylim(0,9)

for axis in range(3):
    # Add vertical lines to show the intraseasonal band
    ax[axis].axvline(x = 1/INTRASEASONAL_LOWCUT, ls=':', lw=4, color='black', alpha=0.5)
    ax[axis].axvline(x = 1/INTRASEASONAL_HIGHCUT, ls=':', lw=4, color='black', alpha=0.5)

    # Add legend
    ax[axis].legend()

    # Configure axes
    ax[axis].set_ylabel('Power')

    for edge in ['top','bottom','left','right']:
        ax[axis].spines[edge].set_linewidth(4)
    ax[axis].tick_params(
        which='major',
        axis='both',
        direction='in',
        top=True,
        right=True,
        width=4,
        length=15,
        color='#bcbcbc',
        pad=10
    )    

    ax[axis].set_xticks(ticks = frequency_from_period)
    ax[axis].xaxis.set_major_formatter(mticker.FixedFormatter(periods))
    ax[axis].set_xlim(1/1000, 1/5)
    ax[axis].set_aspect('auto')
    
ax[-1].set_xlabel('Period (days)')

plt.savefig(
    f"{output_directory}/OLR_zonal_wind_power_spectra_{single_location_longitude:0.0f}E_{single_location_latitude:0.0f}.png",
    dpi=300, 
    bbox_inches='tight'
)

## Compute coherency

In [ ]:
def cohstat(dof,siglev):
    #Siglev, significance level desired, should be .95 or .99. Siglev greater
    #than .95 will default to .99, lower inputs of siglev default to .95

    # this is a rough fit to the F statistic (0.01), assuming the denominator has 100 dof.
    #f=[3.94,3.09,3.98,3.51,3.21,2.99,2.82,2.69,2.59,2.5,2.07,1.89,1.80,1.74,1.65,1.60];
    #n=[1,2,3,4,5,6,7,8,9,10,20,30,40,50,75,100];
    # this is a rough fit to the Coherency statistic (0.01), given dof as input.
    f99 = [0.99,0.684,0.602,0.536,0.482,0.438,0.401,0.342,0.264,0.215,0.175,0.147,0.112,0.075,0.057,0.045,0.023,0.002];
    f90 = [0.901,0.437,0.370,0.319,0.280,0.250,0.226,0.189,0.142,0.112,0.091,0.076,0.057,0.038,0.029,0.023,0.011,0.001];
    f95 = [0.951,0.527,0.450,0.393,0.348,0.312,0.283,0.238,0.181,0.146,0.118,0.098,0.074,0.050,0.037,0.030,0.015,0.001];
    n=[2,5,6,7,8,9,10,12,16,20,25,30,40,60,80,100,200,1000000];

    if  siglev > 0.95:
        f = f99;
    else:
        f = f95
    coh_crit = np.interp(dof,n,f)
    return coh_crit

In [ ]:
frequency_OLR_u850, coherence_OLR_u850 = signal.coherence(
    outgoing_longwave_radiation_standardized, 
    lower_level_zonal_wind_standardized, 
    nperseg=window_size,
    detrend='linear',
    fs=1
)

frequency_OLR_u200, coherence_OLR_u200 = signal.coherence(
    outgoing_longwave_radiation_standardized, 
    upper_level_zonal_wind_standardized, 
    nperseg=window_size,
    detrend='linear',
    fs=1
)

frequency_u850_u200, coherence_u850_u200 = signal.coherence(
    lower_level_zonal_wind_standardized, 
    upper_level_zonal_wind_standardized, 
    nperseg=window_size,
    detrend='linear',
    fs=1
)

critical_coherence = cohstat(degrees_of_freedom_numerator, p_critical)

plt.style.use('bmh')
plt.rcParams.update({'font.size':20})
[fig, ax] = plt.subplots(figsize=(16,9))
ax.set_title(f'Coherence of signals' +
                f' at the point {single_location_longitude:0.0f}°E, {single_location_latitude:0.0f}', 
                pad=15)

# Plot coherence of OLR & U850
ax.plot(
    frequency_OLR_u850,
    coherence_OLR_u850,
    lw=4,
    label='OLR & U850'
)

# Plot coherence of OLR & U200
ax.plot(
    frequency_OLR_u200,
    coherence_OLR_u200,
    lw=4,
    label='OLR & U200'
)


# Plot coherence of U850 & U200
ax.plot(
    frequency_u850_u200,
    coherence_u850_u200,
    lw=4,
    label='U850 & U200'
)

# Plot critical coherence value
ax.axhline(critical_coherence, color='black', lw=4, ls='--', label=f'{100*p_critical:0.0f}% level')

# Plot low and high frequency cutoffs
ax.axvline(x = 1/INTRASEASONAL_LOWCUT, ls=':', lw=4, color='black', alpha=0.5)
ax.axvline(x = 1/INTRASEASONAL_HIGHCUT, ls=':', lw=4, color='black', alpha=0.5)
ax.set_xlabel('Period (Days)')
ax.set_ylabel(r'Coherence$^{2}$')
ax.set_xlim(0, 0.2)
ax.legend(fontsize=12)

for axis in ['top','bottom','left','right']:
    ax.spines[axis].set_linewidth(4)
ax.tick_params(
    which='major',
    axis='both',
    direction='in',
    top=True,
    right=True,
    width=4,
    length=15,
    color='#bcbcbc',
    pad=10
)    
# ax.xaxis.set_major_locator(mticker.MaxNLocator(prune='lower'))
ax.set_xticks(ticks = frequency_from_period)
ax.xaxis.set_major_formatter(mticker.FixedFormatter(periods))
ax.set_xlim(1/1000, 1/9)
ax.set_aspect(9/256)

plt.tight_layout()

# Save figure
plt.savefig(
    f"{output_directory}/OLR_zonal_wind_coherence_{single_location_longitude:0.0f}E_{single_location_latitude:0.0f}.png",
    dpi=300, 
    bbox_inches='tight'
)

In [ ]:
# Calculate cross spectrum of OLR and U850
frequency_cross_spectrum_OLR_u850, cross_spectrum_OLR_u850 = signal.csd(
    outgoing_longwave_radiation_standardized,
    lower_level_zonal_wind_standardized,
    nperseg=window_size,
    detrend='linear',
    fs=1
)

# Calculate phase and covariance
phase_OLR_u850 = np.arctan2(np.imag(cross_spectrum_OLR_u850),np.real(cross_spectrum_OLR_u850))*180/np.pi
covariance_OLR_u850 = np.real(cross_spectrum_OLR_u850)


# Calculate cross spectrum of OLR and U200
frequency_cross_spectrum_OLR_u200, cross_spectrum_OLR_u200 = signal.csd(
    outgoing_longwave_radiation_standardized,
    upper_level_zonal_wind_standardized,
    nperseg=window_size,
    detrend='linear',
    fs=1
)

# Calculate phase and covariance
phase_OLR_u200 = np.arctan2(np.imag(cross_spectrum_OLR_u200),np.real(cross_spectrum_OLR_u200))*180/np.pi
covariance_OLR_u200 = np.real(cross_spectrum_OLR_u200)


# Calculate cross spectrum of U850 and U200
frequency_cross_spectrum_u850_u200, cross_spectrum_u850_u200 = signal.csd(
    lower_level_zonal_wind_standardized,
    upper_level_zonal_wind_standardized,
    nperseg=window_size,
    detrend='linear',
    fs=1
)

# Calculate phase and covariance
phase_u850_u200 = np.arctan2(np.imag(cross_spectrum_u850_u200),np.real(cross_spectrum_u850_u200))*180/np.pi
covariance_u850_u200 = np.real(cross_spectrum_u850_u200)

plt.style.use('bmh')
[fig, ax] = plt.subplots(
    2, 
    1, 
    figsize=(32,18))

# Phase
ax[0].set_title(f'Phase of OLR and U850' +
                f' at the point {single_location_longitude:0.0f}°E, {single_location_latitude:0.0f}', 
                pad=15)

ax[0].plot(
    frequency_cross_spectrum_OLR_u850, 
    phase_OLR_u850, 
    lw=6,
    label='OLR & U850'
)

ax[0].plot(
    frequency_cross_spectrum_OLR_u200, 
    phase_OLR_u200, 
    lw=6,
    label='OLR & U200'
)

ax[0].plot(
    frequency_cross_spectrum_u850_u200, 
    phase_u850_u200, 
    lw=6,
    label='U850 & U200'
)


# Add lines at +/- 90°
ax[0].axhline(y=-90, color='darkgray', ls='--', lw=4)
ax[0].axhline(y=90, color='darkgray', ls='--', lw=4)

# Set phase-specific axis values
ax[0].set_ylim(-185, 185)
ax[0].set_yticks([-180, -135, -90, -45, 0, 45, 90, 135, 180])
ax[0].set_ylabel('Phase')
ax[0].legend()

# Covariance
ax[1].set_title(f'Covariance of OLR and U850' +
                f' at the point {single_location_longitude:0.0f}°E, {single_location_latitude:0.0f}', 
                pad=15)

ax[1].plot(
    frequency_cross_spectrum_OLR_u850, 
    covariance_OLR_u850, 
    lw=6,
    label='OLR & U850'
)

ax[1].plot(
    frequency_cross_spectrum_OLR_u200, 
    covariance_OLR_u200, 
    lw=6,
    label='OLR & U200'
)

ax[1].plot(
    frequency_cross_spectrum_u850_u200, 
    covariance_u850_u200, 
    lw=6,
    label='U850 & U200'
)
        
# Set covariance specific axis values
ax[1].set_ylabel('Covariance')    
ax[1].legend()
      
# Configure general axes
for axis in range(2):
    ax[axis].axvline(x = 1/INTRASEASONAL_LOWCUT, ls=':', lw=4, color='black', alpha=0.5)
    ax[axis].axvline(x = 1/INTRASEASONAL_HIGHCUT, ls=':', lw=4, color='black', alpha=0.5)
    ax[axis].set_xlabel('Period (days)')
    
    ax[axis].set_xlim(0, 0.2)

    for edge in ['top','bottom','left','right']:
        ax[axis].spines[edge].set_linewidth(4)
    ax[axis].tick_params(
        which='major',
        axis='both',
        direction='in',
        top=True,
        right=True,
        width=4,
        length=15,
        color='#bcbcbc',
        pad=10
    )    
    ax[1].xaxis.set_major_locator(mticker.MaxNLocator(prune='lower'))
    ax[axis].set_xticks(ticks = frequency_from_period)
    ax[axis].xaxis.set_major_formatter(mticker.FixedFormatter(periods))
    ax[axis].set_xlim(1/1000, 1/9)

plt.tight_layout()
plt.show()

This analysis clearly shows statistically significant spectra peaks in both OLR and 850-hPa zonal winds in the intraseasonal band. Additionally, there is a significant peak in coherence between the two signals in that band, with a phase relationship around -90° as expected. This is a clear indication of the MJO in the signal. 

# Wheeler and Hendon (2004) Analysis

## Filter Data 

Temporally filter the data on intraseasonal (20-100 day) timescales, using a Lanczos filter

In [ ]:
nyq = 0.5
filter_order = 4
low = (1/INTRASEASONAL_LOWCUT) / nyq
high = (1/INTRASEASONAL_HIGHCUT) / nyq
b, a = signal.butter(filter_order, [low, high], btype="band")

variables_filtered = {}

# for variable in variables_dict:
#     variables_filtered[variable] = variables_deannualized[variable].copy(deep=True)
#     variables_filtered[variable].values = signal.filtfilt(
#         b, a, 
#         variables_deannualized[variable], 
#         axis=0
#     )
    
for variable in variables_dict:
    variables_filtered[variable] = variables_deannualized[variable].copy(deep=True)
    variables_filtered[variable].values = lanczos_bandpass_filter(
    variables_deannualized[variable],
    lowcut=(1 / INTRASEASONAL_LOWCUT),
    highcut=(1 / INTRASEASONAL_HIGHCUT),
    fs=SAMPLING_FREQUENCY,
    filter_axis=0,
    order=241
)
    
variables_filtered['Outgoing Longwave Radiation'] = variables_filtered['Outgoing Longwave Radiation'].isel(time=slice(120, -120)).drop_sel(time=missing_days)

In [ ]:
variables_filtered['Outgoing Longwave Radiation'].sel(lat=slice(-10,10)).mean(dim='lat').sel(time=slice('1992-08-14', '1992-11-12')).plot.contourf(x='lon', levels=np.arange(-30,50,20), extend='both')

In [ ]:
model = xeofs.single.EOF(use_coslat=True, n_modes=2)
model.fit(variables_filtered['Outgoing Longwave Radiation'].sel(lat=slice(-25,25)), dim="time")
EOFs = -model.components()
principle_components = -model.scores()/model.scores().std(dim='time')

In [ ]:
# Set plotting parameters
output_directory = "output/mjo-compositing/"
plt.style.use('default')
plt.rcParams.update({'font.size':24})
cmap_modified = modified_colormap('coolwarm', 'white', 0.1, 0.1)
# cmap_modified = 'BrBG'
coastline_width = 1

fig = plt.figure(figsize=(16,4))
gs = GridSpec(2, 2, height_ratios=[30, 1], figure=fig)
gs.update(top=1, bottom=0, left=0, right=1, hspace=0.3, wspace=0.15)

proj = ccrs.PlateCarree(central_longitude=-180)
data_crs = ccrs.PlateCarree()

ax = [
    fig.add_subplot(gs[0,0], projection=proj),
    fig.add_subplot(gs[0,1], projection=proj)
]

cb_ax = fig.add_subplot(gs[1, :])

# Add cyclic point
cdata = xarray_utils.add_cyclic_point(
    model.scores().std(dim='time')*EOFs,
    dim='lon'
)

for index, mode in enumerate(EOFs.mode):
    # Plot data
    im = ax[index].contourf(
        cdata.lon, 
        cdata.lat, 
        cdata.sel(mode=mode), 
        transform=data_crs,
        cmap=cmap_modified,
        norm=mcolors.CenteredNorm(),
        levels=np.arange(-9, 9+6, 6),
        extend='both'
    )

    ax[index].contour(
        cdata.lon, 
        cdata.lat, 
        cdata.sel(mode=mode), 
        transform=data_crs,
        colors='k',
        levels=np.arange(-18, 18+3, 3)[np.arange(-18, 18+3, 3) != 0],
    )

    # Add colorbar
    cbar = fig.colorbar(im, cax=cb_ax, orientation='horizontal')
    cbar.ax.tick_params(labelsize=20)
    cbar.set_label(r'W m$^{-2}$')

    ax[index].set_aspect('auto')
    ax[index].set_xlabel('')
    # ax.set_global()
    ax[index].add_feature(cf.COASTLINE, lw=coastline_width)

    gl = ax[index].gridlines(
        crs=proj,
        draw_labels=True,
        linewidth=1,
        color="gray",
        alpha=0.75,
        linestyle="-",
        zorder=15

    )
    gl.right_labels = False
    gl.top_labels = False
    gl.xlocator = mticker.FixedLocator(np.arange(-180,180,30))
    # gl.xlocator = LongitudeLocator(30)
    gl.xformatter = LongitudeFormatter()
    gl.xlabel_style = {'fontsize':20}
    gl.ylocator = mticker.FixedLocator(np.arange(-40,40,10))
    gl.yformatter = LatitudeFormatter()
    gl.ylabel_style = {'fontsize':20}

plt.show()
# plt.savefig(f"{output_directory}/time_mean_OLR_zonal_wind_anomalies.png", dpi=300, bbox_inches='tight')

In [ ]:
# Configure plot
plt.style.use('default')
[fig, ax] = plt.subplots(1, 2, figsize=(12,6))
# plt.xlabel("RMM1")
# plt.ylabel("RMM2")

# Plot index points
colormap = sns.color_palette("viridis", as_cmap=True)

# start_time = '1992-02-12' 
# end_time = '1992-05-12'

# start_time = '1992-08-14' 
# end_time = '1992-11-12'

# start_time = np.datetime64("1975-04-01T00:00:00.000000000")
# end_time = np.datetime64("1975-04-30T00:00:00.000000000")

start_time = '1975-02-24T00:00:00.000000000'
end_time = '1975-04-11T00:00:00.000000000'

# for index, (start_time, end_time) in enumerate(zip(('1992-02-12', '1992-08-14'), ('1992-05-12', '1992-11-12'))):
for index, (start_time, end_time) in enumerate(zip((start_time, '1992-08-14'), (end_time, '1992-11-12'))):

    # Plot start_time as empty circle
    ax[index].plot(
        principle_components.sel(mode=1, time=start_time),
        principle_components.sel(mode=2, time=start_time),
        color="black",
        marker="o",
        markerfacecolor='None',
        ls="-",
        ms=15  
    )
    # Plot start_time as empty circle
    # ax[index].scatter(
    #     principle_components.sel(mode=1).where(a, drop=True),
    #     principle_components.sel(mode=2).where(a, drop=True),
    #     color="r",
    # )
    # ax[index].scatter(
    #     principle_components.sel(mode=1).where(b, drop=True),
    #     principle_components.sel(mode=2).where(b, drop=True),
    #     color="blue",
    # )
    # ax[index].scatter(
    #     principle_components.sel(mode=1).where(c, drop=True),
    #     principle_components.sel(mode=2).where(c, drop=True),
    #     color="yellow",
    # )
    # ax[index].scatter(
    #     principle_components.sel(mode=1).where(d, drop=True),
    #     principle_components.sel(mode=2).where(d, drop=True),
    #     color="green",
    # )
    # ax[index].scatter(
    #     principle_components.sel(mode=1).where(n, drop=True),
    #     principle_components.sel(mode=2).where(n, drop=True),
    #     color="purple",
    # )

    # Plot halfway point as filled diamond
    halfway_time = principle_components.sel(time=slice(start_time, end_time)).time.values[len(principle_components.sel(time=slice(start_time, end_time)).time) // 2]
    ax[index].plot(
        principle_components.sel(mode=1, time=halfway_time),
        principle_components.sel(mode=2, time=halfway_time),
        color="black",
        marker="D",
        markerfacecolor='black',
        ls="-",
        ms=15
    )

    ax[index].plot(
        principle_components.sel(mode=1, time=end_time),
        principle_components.sel(mode=2, time=end_time),
        color="black",
        marker="s",
        ls="-",
        ms=15    
    )
    # for time in principle_components.time.sel(time=slice(start_time, end_time)):
    ax[index].plot(
        principle_components.sel(mode=1, time=slice(start_time, end_time)),
        principle_components.sel(mode=2, time=slice(start_time, end_time)),
        color='k',
        linestyle='-',
        marker=".",
        ms=8,
    )

    # Add phase regions overlay
    circle1 = plt.Circle((0, 0), 0.3, color="#bcbcbc", fill=False, lw=1, zorder=10)
    circle2 = plt.Circle((0, 0), 0.4, color="k", fill=False, lw=1.5, zorder=10)
    circle3 = plt.Circle((0, 0), 0.5, color="#bcbcbc", fill=False, lw=1, zorder=10)
    ax[index].add_patch(circle1)
    ax[index].add_patch(circle2)
    ax[index].add_patch(circle3)

    ax[index].axhline(y=0, color="k", lw=1, ls="-")
    ax[index].axvline(x=0, color="k", lw=1, ls="-")

    # Add lines to differentiate the phases
    ax[index].plot(
        [0.4*np.cos(np.deg2rad(45)), 10*np.cos(np.deg2rad(45))],
        [0.4*np.sin(np.deg2rad(45)), 10*np.sin(np.deg2rad(45))],
        color="k",
        lw=1,
        ls="-"
    )
    ax[index].plot(
        [0.4*np.cos(np.deg2rad(45)), 10*np.cos(np.deg2rad(45))],
        [-0.4*np.sin(np.deg2rad(45)), -10*np.sin(np.deg2rad(45))],
        color="k",
        lw=1,
        ls="-"
    )
    ax[index].plot(
        [-0.4*np.cos(np.deg2rad(45)), -10*np.cos(np.deg2rad(45))],
        [-0.4*np.sin(np.deg2rad(45)), -10*np.sin(np.deg2rad(45))],
        color="k",
        lw=1,
        ls="-"
    )
    ax[index].plot(
        [-0.4*np.cos(np.deg2rad(45)), -10*np.cos(np.deg2rad(45))],
        [0.4*np.sin(np.deg2rad(45)), 10*np.sin(np.deg2rad(45))],
        color="k",
        lw=1,
        ls="-"
    )

    ax[index].set_xlim(-3,3)
    ax[index].set_ylim(-3,3)

    # # Add phase labels
    ax[index].text(
        2.9, 0.1,
        f'Category A',
        horizontalalignment='right',
        verticalalignment='center',
        fontsize=12
    )
    ax[index].text(
        0, 2.8,
        f'Category B',
        horizontalalignment='center',
        verticalalignment='center',
        fontsize=12
    )
    ax[index].text(
        -2.9, .1,
        f'Category C',
        horizontalalignment='left',
        verticalalignment='center',
        fontsize=12
    )
    ax[index].text(
        0, -2.8,
        f'Category D',
        horizontalalignment='center',
        verticalalignment='center',
        fontsize=12
    )

    ax[index].spines['left'].set_position('zero')
    ax[index].spines['bottom'].set_position('zero')

    # Hide the top and right spines
    ax[index].spines['right'].set_color('none')
    ax[index].spines['top'].set_color('none')

    # Ensure tick marks follow the spines to the center
    ax[index].xaxis.set_ticks_position('bottom')
    ax[index].yaxis.set_ticks_position('left')

    ax[index].set_aspect("equal")
plt.tight_layout()

In [ ]:
def get_category(principle_components, time):
    if np.sqrt(principle_components.sel(mode=1, time=time)**2 + principle_components.sel(mode=2, time=time)**2) <= 0.3:
        return 'N'
    elif (principle_components.sel(mode=1, time=time) > 0) and (principle_components.sel(mode=1, time=time) > np.abs(principle_components.sel(mode=2, time=time))):
        return 'A'
    elif (principle_components.sel(mode=2, time=time) > 0) and (principle_components.sel(mode=2, time=time) > np.abs(principle_components.sel(mode=1, time=time))):
        return 'B'
    elif (principle_components.sel(mode=1, time=time) < 0) and (-principle_components.sel(mode=1, time=time) > np.abs(principle_components.sel(mode=2, time=time))):
        return 'C'
    elif (principle_components.sel(mode=2, time=time) < 0) and(-principle_components.sel(mode=2, time=time) > np.abs(principle_components.sel(mode=1, time=time))):
        return 'D'


In [ ]:
import itertools

amplitude = np.sqrt(principle_components.sel(mode=1)**2 + principle_components.sel(mode=2)**2)

# start_time = np.datetime64("1990-01-01T00:00:00.000000000")
# end_time = np.datetime64("1990-12-31T00:00:00.000000000")

# start_time = np.datetime64("1975-01-01T00:00:00.000000000")
# end_time = np.datetime64("1975-12-31T00:00:00.000000000")

start_time = principle_components.isel(time=0).time.values
first_end = np.datetime64("1978-03-16")

second_start = np.datetime64("1979-01-01")
end_time = principle_components.isel(time=-1).time.values

# end_time = np.datetime64("1975-12-31T00:00:00.000000000")

# start_time = np.datetime64("1992-08-14T00:00:00.000000000")
# end_time = np.datetime64("1992-11-12T00:00:00.000000000")

time_range_one = np.arange(start_time, first_end, np.timedelta64(1, 'D'))
time_range_two = np.arange(second_start, end_time, np.timedelta64(1, 'D'))

category = []
category.append(get_category(principle_components, principle_components.sel(time=start_time).time))

for i, t in enumerate(itertools.chain(time_range_one, time_range_two)):
    if t == start_time:
        continue

    previous_category = get_category(principle_components, t-np.timedelta64(1, 'D'))
    nominal_category = get_category(principle_components, t)

    if amplitude.sel(time=t) < 0.3:
        category.append('N')
    elif amplitude.sel(time=t) > 0.5:
        category.append(nominal_category)
    else:
        if previous_category == 'N':
            category.append('N')
        else:
            category.append(nominal_category)

    # if previous_category == 'N' and amplitude.sel(time=t) < 0.5:
    #     category.append('N')
    #     # category[i] = 'N'
    # elif previous_category != 'N' and nominal_category == 'N' and amplitude.sel(time=t) > 0.3:
    #     category.append(previous_category)
    #     # category[i] = previous_category
    # else:
    #     category.append(nominal_category)
    #     # category[i] = nominal_category

category = xr.DataArray(
    data=category,
    dims=['time'],
    coords={'time':np.concatenate((time_range_one, time_range_two))}
)

In [ ]:
category

In [ ]:
change_points = np.where(category.values[1:] != category.values[:-1])[0] + 1

collapsed_categories = np.concatenate(([category.values.astype(str)[0]], category.values.astype(str)[change_points]))
s= "".join(collapsed_categories)

pattern = "NABCD"
indices = []

start = 0
while True:
    i = s.find(pattern, start)
    if i == -1:
        break
    indices.append(i)
    start = i + 1   # allow overlapping matches


In [ ]:
import re

category_string = "".join(category.values.astype(str))
pattern = r"N+A+B+C+D+"
matches = list(re.finditer(pattern, category_string))

# transition_indices = [m.start() for m in matches]
# transition_times = category.time[transition_indices]

In [ ]:
test_str = "".join(category[4069:4125].values.astype('str'))
list(re.finditer(pattern, test_str))

In [ ]:
matches

In [ ]:
print(change_points[197])

print(category.isel(time=change_points[0]-1).values)
print(category.isel(time=change_points[0]).values)
print(category.isel(time=change_points[0]+1).values)
print(category.isel(time=change_points[0]+2).values)
print(category.isel(time=change_points[0]+3).values)

In [ ]:
# print(s[indices[0]-1])
# print(s[indices[0]])
# print(s[indices[0]+1])

collapsed_categories[195:205]
# collapsed_categories[197]

In [ ]:
# transition_times

print(get_category(principle_components, np.datetime64("1975-04-16T00:00:00.000000000")))
print(get_category(principle_components, np.datetime64("1975-04-17T00:00:00.000000000")))
print(get_category(principle_components, np.datetime64("1975-04-18T00:00:00.000000000")))

In [ ]:
indices

In [ ]:
get_category(principle_components, principle_components.isel(time=0).time)
for i, t in enumerate(time_range):
    print(f"{amplitude.sel(time=t).values:0.2f}, {category.sel(time=t).values:s}")

In [ ]:
t = principle_components.isel(time=4).time
nominal_category = get_category(principle_components, t)
previous_category = get_category(principle_components, t-np.timedelta64(1, 'D'))
next_category = get_category(principle_components, t+np.timedelta64(1, 'D'))

print(previous_category, nominal_category, next_category)

In [ ]:

matthews = 'CDABCDABCDABCDNABCDANCNADNCBADCNABCDANABC'

In [ ]:
for i in range(0, 10):
    print(t.values)
    t = category.isel(time=i).time
    nominal_category = get_category(principle_components, t)
    previous_category = get_category(principle_components, t-np.timedelta64(1, 'D'))
    next_category = get_category(principle_components, t+np.timedelta64(1, 'D'))

    print(previous_category, nominal_category, next_category, amplitude.sel(time=t).values)

In [ ]:
# amplitude.sel(time="1997-08-17T00:00:00.000000000")
get_category(principle_components, principle_components.sel(time="1997-08-17T00:00:00.000000000").time)

In [ ]:
# category.isel(time=slice(0, 10))
# print(get_category(principle_components, principle_components.time[0]))
# print(principle_components[:,0].values)
category.sel(time="1992-08-14")

In [ ]:
i = 2684
print(get_category(principle_components, i))
print(principle_components.isel(time=i).values)

In [ ]:
principle_components.isel(time=0)

### Plot MJO-filtered data

#### Time mean maps of OLR and 850-hPa winds

In [ ]:
plt.style.use('default')
plt.rcParams.update({'font.size':24})
coastline_width = 1

olr_anomalies = (
    (variables_subset['outgoing longwave radiation']
    - variables_subset['outgoing longwave radiation'].mean(dim=['lat', 'lon'])).mean(dim='time'))


fig = plt.figure(figsize=(16,4))
gs = GridSpec(1, 2, width_ratios=[100,1], figure=fig)
gs.update(top=.95, bottom=0.05, left=0.05, right=.95, hspace=0.6, wspace=0.05)


data_crs = ccrs.PlateCarree()
proj = ccrs.PlateCarree(central_longitude=-205)

ax = fig.add_subplot(gs[0], projection=proj)
cbar_ax = fig.add_subplot(gs[1])

olr_keywords = {
    'transform':data_crs, 
    'cmap':modified_colormap('coolwarm', 'white', 0.05, 0.05),
    'norm':mcolors.CenteredNorm(),
    'levels':21
}

# Plot OLR data
ax.set_title(f"Time-Mean Outgoing Longwave Radiation Anomalies and 850-hPa Zonal Winds", fontsize=20)

# Add OLR cyclic point
cdata, clon = cutil.add_cyclic_point(
    variables_filtered['outgoing longwave radiation'].mean(dim='time'),
    coord=longitude
)

im = ax.contourf(
    clon, 
    latitude, 
    cdata, 
    **olr_keywords
)

cbar = fig.colorbar(im, cax=cbar_ax)
cbar.ax.tick_params(labelsize=20)
cbar.set_label(r'W m$^{-2}$')

# Plot wind data
arrow_spacing = 2
ax.quiver(
    longitude[::2*arrow_spacing],
    latitude[::arrow_spacing],
    variables_filtered['lower level zonal wind'].mean(dim='time')[::arrow_spacing, ::2*arrow_spacing].values,
    variables_filtered['lower level meridional wind'].mean(dim='time')[::arrow_spacing, ::2*arrow_spacing].values,
    transform=data_crs,
    width=0.002,
    scale=0.5
)

# Configure axes and gridlines
ax.set_xlabel('')
ax.add_feature(cf.COASTLINE, lw=coastline_width)
ax.set_aspect('auto')

gl = ax.gridlines(
    crs=proj,
    draw_labels=True,
    linewidth=1,
    color="gray",
    alpha=0.5,
    linestyle="-",
    zorder=15

)
gl.right_labels = False
gl.top_labels = False
gl.xlocator = mticker.FixedLocator(np.arange(-180,180,30))
# gl.xlocator = LongitudeLocator(30)
gl.xformatter = LongitudeFormatter()
gl.xlabel_style = {'fontsize':20}
gl.ylocator = mticker.FixedLocator(np.arange(-30,30,15))
gl.yformatter = LatitudeFormatter()
gl.ylabel_style = {'fontsize':20}

# plt.tight_layout()
plt.show()

#### Hovmoller Diagram

In [ ]:
plt.style.use('default')
plt.rcParams.update({'font.size':24})

# Create figure and gridpsec
fig = plt.figure(figsize=(16,9))
gs = GridSpec(1, 2, width_ratios=[100,2])
gs.update(left=0.05, right=0.95, bottom=0.05, top=0.95, wspace=0.05)

# Specify axes
ax = fig.add_subplot(gs[0])
cbar_ax = fig.add_subplot(gs[1])

# Label plot
ax.set_title(
    f"Hovmoller of Intraseasonally-filtered Precipitation averaged btw. {LATITUDE_NORTH}°S-N", 
    fontsize=24, 
    pad=12
)

# Plot data
im = ax.contourf(
    longitude, 
    np.arange(len(time))[:150],
    variables_filtered['Precipitation'].mean(dim='lat')[:150],
    cmap=modified_colormap('coolwarm', 'white', 0.05, 0.5),
    norm=mcolors.CenteredNorm()
)    

# Add colorbar
cbar = fig.colorbar(im, cax=cbar_ax)
cbar.ax.tick_params(labelsize=20)
cbar.set_label(r'W m$^{-2}$')

# Add line at 180°
ax.axvline(x=180, color='gray', lw=3, ls='--', alpha=0.5)

# Configure axes
ax.set_xticks(np.arange(0,360,30))
ax.set_aspect('auto')
ax.set_xlabel('Longitude')
ax.set_ylabel('Time (days)')

# plt.show()
plt.savefig(
    f"{output_directory}/hovmoller_intraseasonally_filtered_precipitation_30S-30N.png",
    dpi=300, 
    bbox_inches='tight'
)

## Select Boreal Winter 

In [ ]:
# boreal_winter_months = [11,12,1,2,3,4]

# variables_boreal_winter = {}

# for variable in variables_dict:
#     variables_boreal_winter[variable] = variables_filtered[variable].sel(time=
#         variables_filtered[variable]['time.month'].isin(boreal_winter_months)
#     )   

## Average across latitude

In [ ]:
variables_meridional_average = {}

for variable in variables_dict:
    variables_meridional_average[variable] = variables_filtered[variable].copy(deep=True).mean(dim='lat')

## Standardize time series

In [ ]:
variables_standardized = {}

for variable in variables_dict:
    variables_standardized[variable] = variables_meridional_average[variable].copy(deep=True)
    variables_standardized[variable].values = (
        (
            variables_meridional_average[variable] - variables_meridional_average[variable].mean()
        ) / variables_meridional_average[variable].std()
    )

## Concatenate OLR with zonal winds

In [ ]:
combined_data = np.concatenate(
    [
        variables_standardized['outgoing longwave radiation'],
        variables_standardized['upper level zonal wind'],
        variables_standardized['lower level zonal wind'],
    ],
    axis=1,
)

## Compute EOFs

In [ ]:
#### Calculate EOFs and PCs
U, S, VT = np.linalg.svd(combined_data.T, full_matrices=False)
EOF = U.T 
PC = np.dot(np.diag(S), VT) 

# Calculate nominal degrees of freedom
nominal_degrees_of_freedom = np.size(combined_data, 1)

# Calculate eigenvalues and spectrum
eigenvalues = S**2 / nominal_degrees_of_freedom
eigenvalue_spectrum = eigenvalues / np.sum(eigenvalues)  
explained_variance = 100*eigenvalue_spectrum

# Estimate 1-lag autocorrelation and effective degrees of freedom
lag = 1  
B = 0
for k in range(lag - 1, nominal_degrees_of_freedom - lag):
    B = B + np.sum(combined_data[:, k] * combined_data[:, k + lag])
phi_L = 1 / (nominal_degrees_of_freedom - 2 * lag) * B
phi_0 = 1 / nominal_degrees_of_freedom * np.sum(combined_data ** 2)
autocorrelation = phi_L / phi_0
degrees_of_freedom = ((1 - autocorrelation ** 2) / (1 + autocorrelation ** 2))*nominal_degrees_of_freedom

# Estimate uncertainty in eigenvalue spectrum
spectrum_error = eigenvalue_spectrum * np.sqrt(2 / degrees_of_freedom)

# Extract the EOFs of each variable from the array
olr_EOF = EOF[:, :len(longitude)]
upper_level_zonal_wind_EOF = EOF[:, len(longitude) : 2*len(longitude)]
lower_level_zonal_wind_EOF = EOF[:, 2*len(longitude):]

### Plot eigenvalue spectrum

In [ ]:
plt.style.use('bmh')
plt.rcParams.update({'font.size':24})

[fig, ax] = plt.subplots(figsize=(16,9))
index = np.arange(len(eigenvalues))
# ax.set_title(f'Eigenvalue Spectrum, {degrees_of_freedom:0.0f} effective degrees of freedom', pad=15)
ax.errorbar(index, 100*eigenvalue_spectrum, 100*spectrum_error, lw=4)
ax.set_ylabel('Variance Explained')
ax.set_xlabel('Mode')

# Configure and label axes
for axis in ['top','bottom','left','right']:
    ax.spines[axis].set_linewidth(4)

ax.tick_params(
    axis='both', which='major', 
    length=12, width=4, 
    color='#bcbcbc', 
    direction='in',
    right=True,
    top=True,
    pad=5
)

# eof_text =  (f'EOF 1: {explained_variance[0]:0.0f}% of variance \n'
#             +f'EOF 2: {explained_variance[1]:0.0f}% of variance ')
# # Add text showing percentage of explained variance
# ax.text(9.95,0.95, eof_text, fontsize=24,
#        bbox=dict(boxstyle='round', facecolor='#eeeeee', edgecolor='#bcbcbc', alpha=0.5),
#        horizontalalignment='right', verticalalignment='top')

ax.set_ylim(0,100)
ax.set_xlim(-0.05, 10)

plt.tight_layout()
plt.savefig(
    f"{output_directory}/eof-analysis_explained-variance.png",
    dpi=300, 
    bbox_inches='tight'
)

### Plot EOFs

In [ ]:
longitude = outgoing_longwave_radiation.lon
plt.style.use('bmh')
plt.rcParams.update({"font.size": 24})

#%% Plotting
#### EOFs as a function of latitude
[fig, ax] = plt.subplots(2, 1, figsize=(32,18))
fig.suptitle("EOFs of Intraseasonally filtered Tropical data", fontsize=40)

# EOF 1
ax[0].set_title(f"EOF 1, Explained Variance = {explained_variance[0]:0.1f}%", pad=15)
ax[0].plot(
    longitude,
    -olr_EOF[0],
    lw=4,
    label="OLR"
)
ax[0].plot(
    longitude,
    lower_level_zonal_wind_EOF[0],
    ls="--",
    lw=4,
    label="u850"
)
ax[0].plot(
    longitude,
    upper_level_zonal_wind_EOF[0],
    ls="--",
    lw=4,
    label="u200"
)
ax[0].axhline(y=0, lw=2, color="k")

# EOF 2
ax[1].set_title(f"EOF 2, Explained Variance = {explained_variance[1]:0.1f}%", pad=15)
ax[1].plot(
    longitude, 
    olr_EOF[1], 
    lw=4, 
    label="OLR"
)
ax[1].plot(
    longitude,
    -lower_level_zonal_wind_EOF[1],
    ls="--",
    lw=4,
    label="u850"
)

ax[1].plot(
    longitude,
    -upper_level_zonal_wind_EOF[1],
    ls="--",
    lw=4,
    label="u200"
)
ax[1].axhline(y=0, lw=2, color="k")

# Configure axes 
for axis in range(2):
    # Set axis labels and limits
    ax[axis].set_xlabel("Longitude")
    ax[axis].set_ylabel("Normalized Magnitude")
    ax[axis].set_xlim(0, 360)
    ax[axis].set_ylim(-0.2, 0.2)
    # ax[0].set_aspect(360 / 0.8)

    # Specify tick parameters
    ax[axis].xaxis.set_major_formatter(LongitudeFormatter())
    ax[axis].xaxis.set_major_locator(mticker.FixedLocator([0, 60, 120, 180, 240, 300, 360]))
    ax[axis].tick_params(
        which="major",
        width=3,
        length=15,
        direction="in",
        color='#bcbcbc',
        top=True,
        right=True,
        pad=10
    )
    ax[axis].xaxis.set_minor_formatter(LongitudeFormatter())
    ax[axis].xaxis.set_minor_locator(mticker.FixedLocator([30, 90, 150, 210, 270, 330]))
    ax[axis].tick_params(
        which="minor",
        width=3,
        length=7.5,
        direction="in",
        color='#bcbcbc',
        top=True,
        right=True,
        pad=10
    )

    for edge in ['top','bottom','left','right']:
        ax[axis].spines[edge].set_linewidth(4)

    ax[axis].legend(loc="upper right")

plt.tight_layout()
# plt.show()

plt.savefig(
    f"{output_directory}/eof-analysis_eofs-1-2.png",
    dpi=300, 
    bbox_inches='tight'
)

### Plot Power Spectra of PC Time Series

In [ ]:
SEGMENT_LENGTH = 256
OVERLAP = SEGMENT_LENGTH//2
frequency = {}
spectrum = {}

for i in range(1, 4):
    (frequency[i], spectrum[i]) = signal.welch(
        PC[i - 1] - np.mean(PC[i - 1]),
        fs=1,
        window="hann",
        nperseg=SEGMENT_LENGTH,
        noverlap=OVERLAP
    )
    
# Plot the power spectra
plt.style.use('bmh')

[fig, ax] = plt.subplots(figsize=(16,9))
ax.set_title("Power Spectra of Principal Components")
for i in spectrum:
    ax.plot(
        frequency[i],
        spectrum[i],
        label=("PC" + str(i)),
        lw=4
    )

ax.tick_params(
    which='major',
    direction='in',
    color='#bcbcbc',
    length=8,
    width=2,
    pad=15
)

ax.tick_params(
    which='minor',
    direction='in',
    color='#bcbcbc',
    length=4,
    width=1,
    pad=15
)

# Add vertical lines to show the intraseasonal band
ax.axvline(x = 1/100, ls=':', lw=4, color='black', alpha=0.5)
ax.axvline(x = 1/20, ls=':', lw=4, color='black', alpha=0.5)

# Add legend
ax.legend()

# Configure axes
ax.set_ylabel('Power')
ax.set_xlabel('Period (days)')

for edge in ['top','bottom','left','right']:
    ax.spines[edge].set_linewidth(4)
ax.tick_params(
    which='major',
    axis='both',
    direction='in',
    top=True,
    right=True,
    width=4,
    length=15,
    color='#bcbcbc',
    pad=10
)    

ax.set_xticks(ticks = frequency_from_period)
ax.xaxis.set_major_formatter(mticker.FixedFormatter(periods))
ax.set_xlim(1/1000, 1/5)
    
# Set tick parameters
ax.set_xticks(ticks = frequency_from_period)
ax.xaxis.set_major_formatter(mticker.FixedFormatter(periods))
ax.set_xlim(1/1000, 1/5)
    
# Format x-axis
ax.set_xlabel("Period (days)")

ax.legend(loc="best")

plt.tight_layout()

plt.savefig(
    f"{output_directory}/eof-analysis_principal-component_power-spectra.png",
    dpi=300, 
    bbox_inches='tight'
)

## Calculate MJO Phase by RMM

In [ ]:
#### Calculate phases
def compute_mjo_phase_indices(RMM1, RMM2):
    phase_times = {}
    
    # precip_times = precipitation.time.sel(
    # time=slice('1999-01-01T00:00:00.000000000', '2018-12-31T00:00:00.000000000')
    # )
    
    # Find times by phase
    phase_times[1] = time.where((RMM1 < 0) & (RMM2 < 0) & (np.abs(RMM1) > np.abs(RMM2))).dropna(dim='time')
    phase_times[2] = time.where((RMM1 < 0) & (RMM2 < 0) & (np.abs(RMM1) < np.abs(RMM2))).dropna(dim='time')
    phase_times[3] = time.where((RMM1 > 0) & (RMM2 < 0) & (np.abs(RMM1) < np.abs(RMM2))).dropna(dim='time')
    phase_times[4] = time.where((RMM1 > 0) & (RMM2 < 0) & (np.abs(RMM1) > np.abs(RMM2))).dropna(dim='time')
    phase_times[5] = time.where((RMM1 > 0) & (RMM2 > 0) & (np.abs(RMM1) > np.abs(RMM2))).dropna(dim='time')
    phase_times[6] = time.where((RMM1 > 0) & (RMM2 > 0) & (np.abs(RMM1) < np.abs(RMM2))).dropna(dim='time')
    phase_times[7] = time.where((RMM1 < 0) & (RMM2 > 0) & (np.abs(RMM1) < np.abs(RMM2))).dropna(dim='time')
    phase_times[8] = time.where((RMM1 < 0) & (RMM2 > 0) & (np.abs(RMM1) > np.abs(RMM2))).dropna(dim='time')

    # Find indices by phase
    phase_indices = {}
    
    # Find all of the points in phase 1
    phase_indices[1] = np.squeeze(
        np.where((RMM1 < 0) & (RMM2 < 0) & (np.abs(RMM1) > np.abs(RMM2)))
    )

    # Phase 2
    phase_indices[2] = np.squeeze(
        np.where((RMM1 < 0) & (RMM2 < 0) & (np.abs(RMM1) < np.abs(RMM2)))
    )

    # Phase 3
    phase_indices[3] = np.squeeze(
        np.where((RMM1 > 0) & (RMM2 < 0) & (np.abs(RMM1) < np.abs(RMM2)))
    )

    # Phase 4
    phase_indices[4] = np.squeeze(
        np.where((RMM1 > 0) & (RMM2 < 0) & (np.abs(RMM1) > np.abs(RMM2)))
    )

    # Phase 5
    phase_indices[5] = np.squeeze(
        np.where((RMM1 > 0) & (RMM2 > 0) & (np.abs(RMM1) > np.abs(RMM2)))
    )

    # Phase 6
    phase_indices[6] = np.squeeze(
        np.where((RMM1 > 0) & (RMM2 > 0) & (np.abs(RMM1) < np.abs(RMM2)))
    )

    # Phase 7
    phase_indices[7] = np.squeeze(
        np.where((RMM1 < 0) & (RMM2 > 0) & (np.abs(RMM1) < np.abs(RMM2)))
    )

    # Phase 8
    phase_indices[8] = np.squeeze(
        np.where((RMM1 < 0) & (RMM2 > 0) & (np.abs(RMM1) > np.abs(RMM2)))
    )

    return phase_indices, phase_times

## Compute RMM

In [ ]:
# Convert the principal components to an RMM-like index
RMM1 = PC[0] / np.std(PC[0])
RMM2 = -PC[1] / np.std(PC[1])
mjo_strength = np.sqrt(RMM1 ** 2 + RMM2 ** 2)

# Remove weak MJO events
# RMM1[mjo_strength < 1] = np.nan
# RMM2[mjo_strength < 1] = np.nan

RMM1 = xr.DataArray(data=RMM1, coords={'time':time})
RMM2 = xr.DataArray(data=RMM2, coords={'time':time})

[phase_indices, phase_times] = compute_mjo_phase_indices(RMM1, RMM2)


### Plot RMM 

In [ ]:
# Configure plot
[fig, ax] = plt.subplots(figsize=(16,16))
plt.rcParams["axes.edgecolor"] = "black"
plt.rcParams["axes.linewidth"] = 3
ax.set_xlim(-4, 4)
ax.set_ylim(-4, 4)
plt.xlabel("RMM1")
plt.ylabel("RMM2")
ax.set_facecolor("white")

# Plot index points
colormap = sns.color_palette("viridis", as_cmap=True)
start_index = 0
# end_index = len(RMM1) - 1
end_index = 150
ax.plot(RMM1[start_index], RMM2[start_index], color="black", marker=".", ls="-", ms=30)
for i in range(start_index + 1, end_index + 1):
    ax.plot(
        RMM1[i],
        RMM2[i],
        color=colormap((i - start_index) / (end_index - start_index)),
        marker="o",
        ms=10,
    )

# Add phase regions overlay
circle1 = plt.Circle((0, 0), 1.0, color="k", fill=False, lw=3, zorder=10)
ax.hlines(y=0, xmin=-4, xmax=-1, color="k", lw=3, ls="-")
ax.hlines(y=0, xmin=1, xmax=4, color="k", lw=3, ls="-")
ax.vlines(x=0, ymin=-4, ymax=-1, color="k", lw=3, ls="-")
ax.vlines(x=0, ymin=1, ymax=4, color="k", lw=3, ls="-")
ax.plot([np.sqrt(2) / 2, 4], [np.sqrt(2) / 2, 4], color="k", lw=3, ls="-")


x_vals = {
    1:-1,
    2:-1,
    3:1,
    4:1,
    5:1,
    6:1,
    7:-1,
    8:-1
}

y_val1 = {
    1:0,
    2:-4,
    3:-4,
    4:0,
    5:0,
    6:np.linspace(0,4,len(RMM1)),
    7:np.linspace(0,4,len(RMM1)),
    8:0
}

y_val2 = {
    1:-1*np.linspace(0, 4, len(RMM1)),
    2:-1*np.linspace(0, 4, len(RMM1)),
    3:-1*np.linspace(0, 4, len(RMM1)),
    4:-1*np.linspace(0, 4, len(RMM1)),
    5:1*np.linspace(0, 4, len(RMM1)),
    6:4,
    7:4,
    8:1*np.linspace(0, 4, len(RMM1))
}

# Fill one of the phases in with blue
# val=2
# ax.fill_between(
#     x_vals[val]*np.linspace(0, 4, len(RMM1)), 
#     y_val1[val], 
#     y_val2[val]
# )

# x = np.linspace(-1,1,100)
# ax.fill_between(x, -np.sqrt(1-x**2), np.sqrt(1-x**2), color='white')

# Add lines to differentiate the phases
ax.plot([np.sqrt(2) / 2, 4], [-np.sqrt(2) / 2, -4], color="k", lw=3, ls="-")
ax.plot([-4, -np.sqrt(2) / 2], [4, np.sqrt(2) / 2], color="k", lw=3, ls="-")
ax.plot([-4, -np.sqrt(2) / 2], [-4, -np.sqrt(2) / 2], color="k", lw=3, ls="-")
ax.add_patch(circle1)

# Add phase labels
ax.text(-3.5, -0.26, f'Phase 1',  horizontalalignment='center',
     verticalalignment='center')
ax.text(-0.51, -3.75, f'Phase 2', horizontalalignment='center',
     verticalalignment='center')
ax.text(0.5, -3.75, f'Phase 3',   horizontalalignment='center',
     verticalalignment='center')
ax.text(3.5, -0.26, f'Phase 4',   horizontalalignment='center',
     verticalalignment='center')
ax.text(3.5, 0.25, f'Phase 5',    horizontalalignment='center',
     verticalalignment='center')
ax.text(0.5, 3.75, f'Phase 6',    horizontalalignment='center',
     verticalalignment='center')
ax.text(-0.51, 3.75, f'Phase 7',  horizontalalignment='center',
     verticalalignment='center')
ax.text(-3.5, 0.25, f'Phase 8',   horizontalalignment='center',
     verticalalignment='center')

# Add MJO-location labels
# ax.text(0, 3.5, f'Western Pacific',  horizontalalignment='center',
#      verticalalignment='center', bbox=props, fontsize=14)

# ax.text(-3.3, 0, f'Western Hemisphere \n & Africa',  horizontalalignment='center',
#      verticalalignment='center', bbox=props, fontsize=14)

# ax.text(3.3, 0, f'Maritime Continent',  horizontalalignment='center',
#      verticalalignment='center', bbox=props, fontsize=14)

# ax.text(0, -3.5, f'Indian Ocean',  horizontalalignment='center',
#      verticalalignment='center', bbox=props, fontsize=14)

ax.set_aspect("equal")
plt.tight_layout()

plt.savefig(
    f"{output_directory}/eof-analysis_rmm_plot.png",
    dpi=300, 
    bbox_inches='tight'
)

## Plot MJO composites

In [ ]:
# Composite boreal winter rainfall anomalies by MJO phase
rainfall_anomalies_boreal_winter = variables_filtered['precipitation'].where(time['time.month'].isin([11,12,1,2,3,4]))

rainfall_anomalies_by_phase = {}
rainfall_filled = {}

for phase in range(1,9):
#     phase_times[phase] = phase_times[phase].sel(
#     time=slice('1999-01-01T00:00:00.000000000', '2018-12-31T00:00:00.000000000')
# )
    
    rainfall_anomalies_by_phase[phase] = rainfall_anomalies_boreal_winter.sel(time=phase_times[phase]).mean(dim='time')
    # rainfall_filled[phase] = rainfall_anomalies_by_phase[phase].where(np.abs(rainfall_anomalies_by_phase[phase]) >= 0.4)

plt.style.use('default')
plt.rcParams.update({'font.size':24})
plt.rcParams['figure.dpi']= 300
coastline_width = 1
props = dict(boxstyle='round', facecolor='#eeeeee')

data_crs = ccrs.PlateCarree()
proj = ccrs.PlateCarree(central_longitude=-205)

gs = GridSpec(8,2, width_ratios=[100,2])
gs.update(top=.95, bottom=0.05, left=0.05, right=.95, hspace=0.2, wspace=0.075)
fig = plt.figure(figsize=(16, 30))

# cmap_modified = modified_colormap('RdYlBu_r', 'white', 0.05, 0.05)
cmap_modified = modified_colormap('BrBG', 'white', 0.05, 0.05)

keyword_args = {
    'transform':data_crs,
    'cmap':cmap_modified,
    'levels':np.linspace(-4, 4, 41),
    'norm':mcolors.CenteredNorm(vcenter=0, halfrange=4),
    'extend':'both'
}

for phase in range(1,9):
    # Add an axis object for each phase
    ax = fig.add_subplot(gs[phase-1, 0], projection=proj)
    
    ax.set_title(f"Phase {phase:0.0f}")
    
    # Add the cyclic point
    cdata, clon = cutil.add_cyclic_point(
        rainfall_anomalies_by_phase[phase],
        coord=longitude
    )
    
    # Plot rainfall by phase
    im = ax.contourf(
        clon, 
        latitude,
        cdata,
        **keyword_args
    )    

    # Map parameters
    ax.set_xlabel('')
    ax.add_feature(cf.COASTLINE, lw=coastline_width)

    gl = ax.gridlines(
        crs=proj,
        draw_labels=True,
        linewidth=1,
        color="gray",
        alpha=0.5,
        linestyle="-",

    )
    gl.right_labels = False
    gl.top_labels = False
    gl.xlocator = mticker.FixedLocator(np.arange(-180,180,30))
    # gl.xlocator = LongitudeLocator(30)
    gl.xformatter = LongitudeFormatter()
    gl.xlabel_style = {'fontsize':20}
    # gl.ylocator = mticker.FixedLocator(np.arange(-15,15,5))
    # gl.yformatter = LatitudeFormatter()
    # gl.ylabel_style = {'fontsize':20}
        
    # # Label the phase in the top left corner
    # ax.text(
    #     0.015, 
    #     0.925, 
    #     f'Phase {phase:0.0f}', 
    #     horizontalalignment='left',
    #     verticalalignment='top', 
    #     transform=ax.transAxes, 
    #     bbox=props
    # )
    
    ax.set_aspect('auto')
    
# Set colorbar
cbar_ax = fig.add_subplot(gs[2:-2,1])
cbar = fig.colorbar(im, cax=cbar_ax)
cbar.ax.tick_params(labelsize=20)
cbar.set_label(r'mm day$^{-1}$')
    
plt.savefig(
    f"{output_directory}/mjo_composited_by_phase.png",
    dpi=300, 
    bbox_inches='tight'
)

### Animation


In [ ]:
plt.style.use('default')
plt.rcParams.update({'font.size':24})
plt.rcParams['figure.dpi']= 300
coastline_width = 1
props = dict(boxstyle='round', facecolor='#eeeeee')

data_crs = ccrs.PlateCarree()
proj = ccrs.PlateCarree(central_longitude=-205)

gs = GridSpec(1,2, width_ratios=[100,2])
gs.update(left=0.1, right=.9, bottom=0.1, top=.9, hspace=0.15, wspace=0.05)
fig = plt.figure(figsize=(16, 4))

# cmap_modified = modified_colormap('RdYlBu_r', 'white', 0.05, 0.05)
cmap_modified = modified_colormap('BrBG', 'white', 0.05, 0.05)

keyword_args = {
    'transform':data_crs,
    'cmap':cmap_modified,
    'levels':np.linspace(-4, 4, 41),
    'norm':mcolors.CenteredNorm(vcenter=0, halfrange=4),
    'extend':'both'
}

# Add an axis object for each phase
ax = fig.add_subplot(gs[0], projection=proj)
cbar_ax = fig.add_subplot(gs[1])

# # inset_ax = plt.axes([0.03, 0.7, 0.3, 0.3])
# inset_x = 0.2
# inset_y = 0.01
# inset_width = 0.3
# inset_height = 0.3

# # Add the inset to the figure
# inset_ax = ax.inset_axes([inset_x, 1-inset_y-inset_height, inset_width, inset_height])
# inset_ax.set_aspect('equal')

# # Add phase regions overlay
# circle1 = plt.Circle((0, 0), 1.0, color="k", fill=False, lw=1, zorder=10)
# inset_ax.hlines(y=0, xmin=-4, xmax=-1, color="k", lw=1, ls="-")
# inset_ax.hlines(y=0, xmin=1, xmax=4, color="k", lw=1, ls="-")
# inset_ax.vlines(x=0, ymin=-4, ymax=-1, color="k", lw=1, ls="-")
# inset_ax.vlines(x=0, ymin=1, ymax=4, color="k", lw=1, ls="-")
# inset_ax.plot([np.sqrt(2) / 2, 4], [np.sqrt(2) / 2, 4], color="k", lw=1, ls="-")
# inset_ax.set_xticks([])
# inset_ax.set_yticks([])

# x_vals = {
#     1:-1,
#     2:-1,
#     3:1,
#     4:1,
#     5:1,
#     6:1,
#     7:-1,
#     8:-1
# }

# y_val1 = {
#     1:0,
#     2:-4,
#     3:-4,
#     4:0,
#     5:0,
#     6:np.linspace(0,4,len(RMM1)),
#     7:np.linspace(0,4,len(RMM1)),
#     8:0
# }

# y_val2 = {
#     1:-1*np.linspace(0, 4, len(RMM1)),
#     2:-1*np.linspace(0, 4, len(RMM1)),
#     3:-1*np.linspace(0, 4, len(RMM1)),
#     4:-1*np.linspace(0, 4, len(RMM1)),
#     5:1*np.linspace(0, 4, len(RMM1)),
#     6:4,
#     7:4,
#     8:1*np.linspace(0, 4, len(RMM1))
# }

# val=1
# x = np.linspace(-1,1,100)
# fill = inset_ax.fill_between(
#     x_vals[val]*np.linspace(0, 4, len(RMM1)), 
#     y_val1[val], 
#     y_val2[val]
# )
# fill2 = inset_ax.fill_between(x, -np.sqrt(1-x**2), np.sqrt(1-x**2), color='white')


# inset_ax.plot([np.sqrt(2) / 2, 4], [-np.sqrt(2) / 2, -4], color="k", lw=1, ls="-")
# inset_ax.plot([-4, -np.sqrt(2) / 2], [4, np.sqrt(2) / 2], color="k", lw=1, ls="-")
# inset_ax.plot([-4, -np.sqrt(2) / 2], [-4, -np.sqrt(2) / 2], color="k", lw=1, ls="-")
# inset_ax.add_patch(circle1)

# # # Add phase labels
# inset_ax.text(-3.5, -0.26, f'1',  horizontalalignment='center',
#      verticalalignment='center', fontsize=6)
# inset_ax.text(-0.51, -3.75, f'2', horizontalalignment='center',
#      verticalalignment='center', fontsize=6)
# inset_ax.text(0.5, -3.75, f'3',   horizontalalignment='center',
#      verticalalignment='center', fontsize=6)
# inset_ax.text(3.5, -0.26, f'4',   horizontalalignment='center',
#      verticalalignment='center', fontsize=6)
# inset_ax.text(3.5, 0.25, f'5',    horizontalalignment='center',
#      verticalalignment='center', fontsize=6)
# inset_ax.text(0.5, 3.75, f'6',    horizontalalignment='center',
#      verticalalignment='center', fontsize=6)
# inset_ax.text(-0.51, 3.75, f'7',  horizontalalignment='center',
#      verticalalignment='center', fontsize=6)
# inset_ax.text(-3.5, 0.25, f'8',   horizontalalignment='center',
#      verticalalignment='center', fontsize=6)


ax.set_title(f"Phase {1:0.0f}")

# Add the cyclic point
cdata, clon = cutil.add_cyclic_point(
    rainfall_anomalies_by_phase[phase],
    coord=longitude
)

ax._autoscaleXon = False
ax._autoscaleYon = False

# Plot rainfall by phase
im = ax.contourf(
    clon, 
    latitude,
    cdata,
    **keyword_args
)    

# Set colorbar
cbar = fig.colorbar(im, cax=cbar_ax)
cbar.ax.tick_params(labelsize=20)
cbar.set_label(r'mm day$^{-1}$')

# Map parameters
ax.set_xlabel('')
ax.add_feature(cf.COASTLINE, lw=coastline_width)

gl = ax.gridlines(
    crs=proj,
    draw_labels=False,
    linewidth=1,
    color="gray",
    alpha=0.5,
    linestyle=":"
)
# gl.right_labels = False
# gl.top_labels = False
# gl.xlocator = mticker.FixedLocator(np.arange(-180,180,30))
# # gl.xlocator = LongitudeLocator(30)
# gl.xformatter = LongitudeFormatter()
# gl.xlabel_style = {'fontsize':20}
# # gl.ylocator = mticker.FixedLocator(np.arange(-15,15,5))
# # gl.yformatter = LatitudeFormatter()
# # gl.ylabel_style = {'fontsize':20}

# # Label the phase in the top left corner
# text= ax.text(
#     0.015, 
#     0.925, 
#     f'Phase {1:0.0f}', 
#     horizontalalignment='left',
#     verticalalignment='top', 
#     transform=ax.transAxes, 
#     bbox=props
# )

ax.set_aspect('auto')

def update(frame):
    ax.set_title(f"Phase {frame+1:0.0f}")

    # Add the cyclic point
    cdata, clon = cutil.add_cyclic_point(
        rainfall_anomalies_by_phase[frame+1],
        coord=longitude
    )
    ax._autoscaleXon = False
    ax._autoscaleYon = False

    # Plot rainfall by phase
    im = ax.contourf(
        clon, 
        latitude,
        cdata,
        **keyword_args
    )    
        
#     text = ax.text(
#         0.015, 
#         0.925, 
#         f'Phase {frame+1:0.0f}', 
#         horizontalalignment='left',
#         verticalalignment='top', 
#         transform=ax.transAxes, 
#         bbox=props
# )
        
    # Map parameters
    ax.set_xlabel('')
    ax.add_feature(cf.COASTLINE, lw=coastline_width)

#     fill = inset_ax.fill_between(
#         x_vals[frame+1]*np.linspace(0, 4, len(RMM1)), 
#         y_val1[frame+1], 
#         y_val2[frame+1],
#         color='#1f77b4'
#     )
    
#     if frame != 0:
#         fill = inset_ax.fill_between(
#         x_vals[frame]*np.linspace(0, 4, len(RMM1)), 
#         y_val1[frame], 
#         y_val2[frame],
#         color='white'
#         )
#     fill2 = inset_ax.fill_between(x, -np.sqrt(1-x**2), np.sqrt(1-x**2), color='white')
    
    return im, #fill, fill2

# Create the animation
animation = FuncAnimation(fig, update, frames=tqdm(
        # np.arange(0, 8, 1), 
        np.array([0,1,2,3,4,5,6,7,7,7]),
        ncols=100, 
        position=0, 
        leave=True
    ), interval=550)

animation.save(f"{output_directory}/mjo_phases_BrBG.gif", dpi=300)